# Notebook 2: Temporal Analysis & Feature Engineering
    
This notebook analyzes the temporal retention windows (clicks vs orders) and builds the dataset required for the LambdaMART and Attention models.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

BASE = Path(r"C:\Users\Acer\OneDrive - National University of Singapore\BT4222\JDsearch")
ARTIFACT_DIR = BASE / "artifacts"
PROCESSED_DIR = BASE / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

cand_df = pd.read_parquet(ARTIFACT_DIR / "candidate_interactions.parquet")
hist_df = pd.read_parquet(ARTIFACT_DIR / "historical_interactions.parquet")

print("Loaded data.")
print(f"Candidates: {cand_df.shape} | Histories: {hist_df.shape}")

Loaded data.
Candidates: (15510012, 6) | Histories: (26495513, 6)


## 1. Temporal Retention Analysis
We map how far back each interaction type exists in the history relative to the test query.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import gc

BASE = Path(r"C:\Users\Acer\OneDrive - National University of Singapore\BT4222\JDsearch")
ARTIFACT_DIR = BASE / "artifacts"
PROCESSED_DIR = BASE / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Loading data (Memory Optimized)...")

# Load candidates
cand_df = pd.read_parquet(
    ARTIFACT_DIR / "candidate_interactions.parquet", 
    columns=["session_id", "item_id", "raw_action_label"]
)
# Drop rows where item_id is missing (safety)
cand_df = cand_df.dropna(subset=["item_id"])
cand_df["session_id"] = cand_df["session_id"].astype("int32")
cand_df["item_id"] = cand_df["item_id"].astype("int32")
cand_df["raw_action_label"] = cand_df["raw_action_label"].astype("float32")

# Load histories
hist_df = pd.read_parquet(
    ARTIFACT_DIR / "historical_interactions.parquet", 
    columns=["session_id", "item_id", "action_type", "time_to_next"]
)
# Drop rows where item_id is missing (safety)
hist_df = hist_df.dropna(subset=["item_id"])
hist_df["session_id"] = hist_df["session_id"].astype("int32")
hist_df["item_id"] = hist_df["item_id"].astype("int32")
hist_df["action_type"] = hist_df["action_type"].astype("category")
hist_df["time_to_next"] = pd.to_numeric(hist_df["time_to_next"], errors="coerce").fillna(0.0).astype("float32")

gc.collect()

print("Loaded data.")
print(f"Candidates: {cand_df.shape} | Histories: {hist_df.shape}")

Loading data (Memory Optimized)...
Loaded data.
Candidates: (15510012, 3) | Histories: (26493429, 4)


## 2. Build Train/Val/Test Splits
We split by session_id (user) to ensure no leakage.

In [4]:
print("Splitting data by session_id (70/15/15)...")
unique_sessions = cand_df["session_id"].unique()
rng = np.random.default_rng(42)
rng.shuffle(unique_sessions)

n_train = int(0.70 * len(unique_sessions))
n_val = int(0.15 * len(unique_sessions))

train_sessions = unique_sessions[:n_train]
val_sessions = unique_sessions[n_train:n_train+n_val]
test_sessions = unique_sessions[n_train+n_val:]

cand_df["split"] = "test"
cand_df.loc[cand_df["session_id"].isin(train_sessions), "split"] = "train"
cand_df.loc[cand_df["session_id"].isin(val_sessions), "split"] = "val"

hist_df["split"] = "test"
hist_df.loc[hist_df["session_id"].isin(train_sessions), "split"] = "train"
hist_df.loc[hist_df["session_id"].isin(val_sessions), "split"] = "val"

print(f"Train: {len(train_sessions)} sessions | Val: {len(val_sessions)} | Test: {len(test_sessions)}")
cand_df.to_parquet(PROCESSED_DIR / "candidates_split.parquet", index=False)
hist_df.to_parquet(PROCESSED_DIR / "history_split.parquet", index=False)

Splitting data by session_id (70/15/15)...
Train: 121681 sessions | Val: 26074 | Test: 26076
